# Urdu Story Preprocessing Pipeline
## For BPE Tokenizer → Trigram Language Model

**Data sources:** `adabiduniya.json` (76), `milkystory.json` (192), `urdupoint.json` (175) = **443 stories**

### Pipeline Steps:
1. Load & explore raw data
2. Text cleaning (remove scraping artifacts, HTML leftovers, promotional noise)
3. Unicode normalization (critical for Urdu — ی/ي, ک/ك, ہ/ھ)
4. Quotation normalization
5. Whitespace & structural normalization
6. Special token injection (`<BOS>`, `<EOS>`, `<PARA>`, `<NL>`)
7. Combine & save final corpus

In [21]:
import json
import re
import unicodedata
import os
from pathlib import Path
from collections import Counter

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../data")

# Load all data files
def load_json(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)

adabi = load_json(DATA_DIR / "adabiduniya.json")
milky = load_json(DATA_DIR / "milkystory.json")
urdupoint = load_json(DATA_DIR / "urdupoint.json")

print(f"adabiduniya: {len(adabi)} stories")
print(f"milkystory:  {len(milky)} stories")
print(f"urdupoint:   {len(urdupoint)} stories")
print(f"Total:       {len(adabi) + len(milky) + len(urdupoint)} stories")

adabiduniya: 78 stories
milkystory:  192 stories
urdupoint:   843 stories
Total:       1113 stories


## Step 1: Explore Raw Data — Identify Noise Patterns

In [22]:
# Explore noise patterns across sources
def show_samples(data, source_name, n=3):
    print(f"\n{'='*60}")
    print(f"  {source_name} — First {n} content samples (first 300 chars)")
    print(f"{'='*60}")
    for i, story in enumerate(data[:n]):
        print(f"\n--- Story {i+1}: {story['title']} ---")
        print(repr(story['content'][:300]))
        print()

show_samples(adabi, "adabiduniya")
show_samples(milky, "milkystory")
show_samples(urdupoint, "urdupoint")


  adabiduniya — First 3 content samples (first 300 chars)

--- Story 1: دودھ گر گیا: ناکامی سے اٹھنے کی ایک دل کو چھو لینے والی موٹیویشنل کہانی ---
'ایک چھوٹے سے گاؤں میں، سبز پہاڑیوں اور بہتی ندیوں کے درمیان، روہن نامی ایک 10 سال کا لڑکا اپنی بیوہ ماں کاملا کے ساتھ ایک مٹی کے چھوٹے سے گھر میں رہتا تھا۔ کاملا روزانہ امیر گھرانوں کے لیے کپڑے دھوتی، اس کے ہاتھ سخت اور تھکے ہوئے تھے، مگر روہن سے اس کی محبت کبھی کم نہ ہوتی۔ دونوں کے پاس بہت کم تھا، '


--- Story 2: The Tortoise and the Hare Story In Urdu A Timeless Tale of Patience and Perseverance ---
'یہ ویڈیو ایک کلاسک Aesop کی کہانی کا خوبصورت اینیمیٹڈ ری ٹیلنگ ہے جو صبر، ہمت، تواضع اور مستقل مزاجی کا سبق دیتی ہے۔ یہ بچوں اور فیملی کے لیے آرام دہ، سکون بخش اور اخلاقی سبق سے بھری کہانی ہے جو انگریزی میں ہے (بچوں کی انگریزی سیکھنے کے لیے بھی بہترین)۔\n\nایک سرسبز جنگل میں Leo the Hare (خرگوش) بہت ف'


--- Story 3: بڑا سوچنے کی طاقت: خالی جیبوں والے لڑکے کی زندگی بدل دینے والی کہانی ---
'ایک چھوٹے سے گاؤں میں ایک غریب لڑکا آرو رہتا تھا۔ 

In [23]:
# Detect common noise patterns in content
def analyze_noise(data, source_name):
    print(f"\n{'='*40} {source_name} {'='*40}")
    
    english_heavy = 0
    has_toc = 0
    has_promo = 0
    has_category_tags = 0
    has_date_stamps = 0
    total_chars = 0
    
    for story in data:
        c = story['content']
        total_chars += len(c)
        
        # English-heavy check: count English chars vs total
        eng_chars = len(re.findall(r'[a-zA-Z]', c))
        if eng_chars / max(len(c), 1) > 0.3:
            english_heavy += 1
        
        # Table of contents pattern
        if 'فہرستِ عنوانات' in c or 'فہرست' in c:
            has_toc += 1
        
        # Promotional footers
        if any(p in c for p in ['شیئر ضرور کریں', 'مزید پڑھیں', 'آپ کا خیال؟', 'کمٹ میں بتائیں']):
            has_promo += 1
        
        # Category tags (milkystory pattern)
        if re.search(r'SHOWBIZ|FEATURED|انٹرٹینمٹ|منٹ پڑھیں', c):
            has_category_tags += 1
        
        # Date stamps
        if re.search(r'\d{1,2}/\d{1,2}/\d{4}', c):
            has_date_stamps += 1
    
    print(f"  Total stories:      {len(data)}")
    print(f"  Avg content length: {total_chars // len(data)} chars")
    print(f"  English-heavy (>30% eng chars): {english_heavy}")
    print(f"  Has TOC header:     {has_toc}")
    print(f"  Has promo footer:   {has_promo}")
    print(f"  Has category tags:  {has_category_tags}")
    print(f"  Has date stamps:    {has_date_stamps}")

analyze_noise(adabi, "adabiduniya")
analyze_noise(milky, "milkystory")
analyze_noise(urdupoint, "urdupoint")


======================================== adabiduniya ========================================
  Total stories:      78
  Avg content length: 2697 chars
  English-heavy (>30% eng chars): 0
  Has TOC header:     17
  Has promo footer:   3
  Has category tags:  0
  Has date stamps:    0

======================================== milkystory ========================================
  Total stories:      192
  Avg content length: 2499 chars
  English-heavy (>30% eng chars): 0
  Has TOC header:     2
  Has promo footer:   0
  Has category tags:  112
  Has date stamps:    102

======================================== urdupoint ========================================
  Total stories:      843
  Avg content length: 756 chars
  English-heavy (>30% eng chars): 0
  Has TOC header:     1
  Has promo footer:   0
  Has category tags:  0
  Has date stamps:    0


In [24]:
# Check for Unicode inconsistencies (Arabic vs Urdu forms)
def check_unicode_variants(data, source_name):
    arabic_yeh = 0   # ي (U+064A) vs ی (U+06CC)
    arabic_kaf = 0   # ك (U+0643) vs ک (U+06A9)
    arabic_heh = 0   # ھ (U+06BE) vs ہ (U+06C1)
    zwj_count = 0    # Zero-width joiners
    zwnj_count = 0   # Zero-width non-joiners
    
    for story in data:
        c = story['content']
        arabic_yeh += c.count('\u064A')   # ي
        arabic_kaf += c.count('\u0643')   # ك
        arabic_heh += c.count('\u06BE')   # ھ (do-chashmi he)
        zwj_count += c.count('\u200D')    # ZWJ
        zwnj_count += c.count('\u200C')   # ZWNJ
    
    print(f"\n{'='*40} {source_name} {'='*40}")
    print(f"  Arabic Yeh (ي U+064A): {arabic_yeh}")
    print(f"  Arabic Kaf (ك U+0643): {arabic_kaf}")  
    print(f"  Do-chashmi He (ھ U+06BE): {arabic_heh}")
    print(f"  Zero-width joiners (ZWJ):  {zwj_count}")
    print(f"  Zero-width non-joiners (ZWNJ): {zwnj_count}")

check_unicode_variants(adabi, "adabiduniya")
check_unicode_variants(milky, "milkystory")
check_unicode_variants(urdupoint, "urdupoint")


======================================== adabiduniya ========================================
  Arabic Yeh (ي U+064A): 8
  Arabic Kaf (ك U+0643): 0
  Do-chashmi He (ھ U+06BE): 3942
  Zero-width joiners (ZWJ):  0
  Zero-width non-joiners (ZWNJ): 0

======================================== milkystory ========================================
  Arabic Yeh (ي U+064A): 1
  Arabic Kaf (ك U+0643): 0
  Do-chashmi He (ھ U+06BE): 8729
  Zero-width joiners (ZWJ):  0
  Zero-width non-joiners (ZWNJ): 0

======================================== urdupoint ========================================
  Arabic Yeh (ي U+064A): 0
  Arabic Kaf (ك U+0643): 0
  Do-chashmi He (ھ U+06BE): 16836
  Zero-width joiners (ZWJ):  0
  Zero-width non-joiners (ZWNJ): 0


## Step 2: Define the Preprocessing Pipeline

The full pipeline in order:
1. **Remove scraping artifacts** — TOC headers, category tags, date stamps, promo footers, "منٹ پڑھیں" patterns
2. **Remove/clean English noise** — English sentences/phrases that aren't meaningful for Urdu modeling (keep transliterated Urdu words)
3. **Unicode NFC normalization** — canonical composition
4. **Urdu character normalization** — Arabic→Urdu form mappings (ي→ی, ك→ک, etc.)
5. **Clean zero-width characters** — remove inconsistent ZWJ/ZWNJ
6. **Normalize quotation marks** — all variants → `"`
7. **Normalize whitespace** — collapse spaces/tabs, normalize newlines, preserve paragraph breaks
8. **Inject special tokens** — `<BOS>`, `<EOS>`, `<PARA>`, `<NL>`
9. **Final quality filter** — drop stories that are too short or too English-heavy after cleaning

In [25]:
# ============================================================
#  STEP 2A: Scraping Artifact Removal (source-specific)
# ============================================================

def remove_adabi_artifacts(text):
    """Remove adabiduniya.com specific artifacts."""
    # Remove TOC header block (فہرستِ عنوانات followed by list of section names)
    text = re.sub(r'^فہرستِ?\s*عنوانات\n+', '', text)
    
    # Remove promotional footers
    # Pattern: "اگر یہ مضمون پسند آیا تو شیئر ضرور کریں..." to end
    text = re.sub(r'اگر یہ مضمون پسند آیا تو شیئر ضرور کریں.*$', '', text, flags=re.DOTALL)
    
    # Remove "مزید پڑھیں ادبی دنیا پر:" sections at the end
    text = re.sub(r'مزید پڑھیں ادبی دنیا پر:.*$', '', text, flags=re.DOTALL)
    
    # Remove "آج سے ایک چھوٹا قدم اٹھائیں" CTA blocks
    text = re.sub(r'آج سے ایک چھوٹا قدم اٹھائیں.*?(?=\n\n|\Z)', '', text, flags=re.DOTALL)
    
    # Remove "آپ کا خیال؟" prompt blocks
    text = re.sub(r'آپ کا خیال؟.*?(?=\n\n|\Z)', '', text, flags=re.DOTALL)
    
    return text


def remove_milky_artifacts(text):
    """Remove milkystory.com specific artifacts."""
    # Remove category/tag headers like "SHOWBIZانٹرٹینمٹ/شوبزFEATURED" etc
    text = re.sub(r'^(?:SHOWBIZ|FEATURED|انٹرٹینمٹ/شوبز|خبریں|اردو کہانیاں|انٹرٹینمٹ|شوبز)+[\s/]*\n*', '', text)
    
    # Remove date + read time stamps like "12/21/20241 منٹ پڑھیں"
    text = re.sub(r'\d{1,2}/\d{1,2}/\d{4}\s*\d*\s*منٹ پڑھیں\s*', '', text)
    
    # Also standalone dates at beginning
    text = re.sub(r'^\d{1,2}/\d{1,2}/\d{4}\s*', '', text)
    
    return text


def remove_urdupoint_artifacts(text):
    """Remove urdupoint.com specific artifacts (minimal — cleanest source)."""
    # urdupoint is the cleanest, no major artifacts observed
    return text


# Source-specific English title/subtitle patterns in adabiduniya
def remove_english_titles(text):
    """Remove standalone English titles/subtitles embedded in Urdu content."""
    # Pattern: English text between Urdu text (common in adabiduniya)
    # e.g., "The Untold Story of Ramadan –", "The Great Wall of China –"
    # Remove lines that are purely or predominantly English
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            cleaned_lines.append(line)
            continue
        # Count proportion of English alphabetic chars
        eng_chars = len(re.findall(r'[a-zA-Z]', stripped))
        total_chars = len(stripped.replace(' ', ''))
        if total_chars > 0 and eng_chars / total_chars > 0.6 and len(stripped) > 10:
            # Skip predominantly English lines (>60% English, >10 chars)
            continue
        cleaned_lines.append(line)
    return '\n'.join(cleaned_lines)

print("✓ Artifact removal functions defined")

✓ Artifact removal functions defined


In [26]:
# ============================================================
#  STEP 2B: Unicode & Urdu Character Normalization
# ============================================================

def normalize_unicode(text):
    """Apply NFC normalization — canonical composition.
    This ensures composed forms are used consistently."""
    return unicodedata.normalize("NFC", text)


def normalize_urdu_chars(text):
    """Normalize Arabic character variants to standard Urdu forms.
    Critical for BPE — prevents vocab explosion from duplicate characters."""
    replacements = {
        '\u064A': '\u06CC',  # ي (Arabic Yeh) → ی (Urdu Yeh)
        '\u0643': '\u06A9',  # ك (Arabic Kaf) → ک (Urdu Kaf)
        # NOTE: ھ (U+06BE do-chashmi he) is VALID in Urdu for aspirated consonants
        # like بھ, تھ, پھ, جھ, دھ, ڈھ, گھ, کھ, لھ, مھ, نھ
        # So we do NOT blindly replace ھ→ہ — that would destroy words like "بھی" → "بہی"
        # Instead, we only normalize standalone ھ when it appears as the last letter
        # and is not part of an aspiration pair.
        
        '\u0629': '\u06C3',  # ة (Arabic Ta Marbuta) → ۃ (Urdu)
        '\u06C0': '\u06C1\u0654',  # ۀ → ۂ (Hamza above He)
        
        # Alef variants
        '\u0622': '\u0622',  # آ stays as آ (Alef with Madda)
        '\u0623': '\u0627',  # أ → ا (Alef with Hamza above → plain Alef)
        '\u0625': '\u0627',  # إ → ا (Alef with Hamza below → plain Alef)
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text


def clean_zero_width(text):
    """Remove zero-width joiners/non-joiners that cause inconsistency.
    ZWNJ (U+200C) is sometimes used meaningfully in Urdu for compound words,
    but scraped data has it inconsistently. Remove for BPE consistency."""
    text = text.replace('\u200D', '')   # ZWJ (zero-width joiner)
    text = text.replace('\u200C', '')   # ZWNJ (zero-width non-joiner)
    text = text.replace('\u200B', '')   # ZWSP (zero-width space)
    text = text.replace('\u200E', '')   # LRM (left-to-right mark) 
    text = text.replace('\u200F', '')   # RLM (right-to-left mark)
    text = text.replace('\uFEFF', '')   # BOM
    return text

print("✓ Unicode normalization functions defined")

✓ Unicode normalization functions defined


In [27]:
# ============================================================
#  STEP 2C: Text Cleaning — Punctuation & Quotation Normalization
# ============================================================

def normalize_quotations(text):
    """Normalize all quotation mark variants to standard double quote.
    Urdu text uses a mix of «», "", "", '' — normalize to " for consistency."""
    # Smart/curly quotes → straight
    text = text.replace('\u201C', '"')  # "
    text = text.replace('\u201D', '"')  # "
    text = text.replace('\u2018', "'")  # '
    text = text.replace('\u2019', "'")  # '
    text = text.replace('\u00AB', '"')  # «
    text = text.replace('\u00BB', '"')  # »
    text = text.replace('\u201E', '"')  # „
    return text


def clean_punctuation(text):
    """Clean and normalize punctuation while preserving meaningful ones.
    
    KEEP: ۔ ، ؟ ! : ؛ — … " ( ) -
    REMOVE: stray special chars, multiple punctuation, etc.
    """
    # Normalize dashes: em-dash, en-dash → single em-dash
    text = text.replace('\u2013', '—')  # en-dash → em-dash
    text = text.replace('\u2014', '—')  # em-dash stays
    text = text.replace('--', '—')
    
    # Normalize ellipsis
    text = re.sub(r'\.{3,}', '…', text)  # ... → …
    text = re.sub(r'۔{2,}', '…', text)   # multiple Urdu full stops → ellipsis
    
    # Remove stray asterisks, hash marks, etc
    text = re.sub(r'[#*_~`|]', '', text)
    
    # Remove stray HTML entities if any
    text = re.sub(r'&\w+;', '', text)
    text = re.sub(r'<[^>]+>', '', text)  # Remove any HTML tags
    
    return text


def remove_emojis(text):
    """Remove emoji characters — not useful for Urdu language modeling."""
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "\U0001f926-\U0001f937"
        "\U00010000-\U0010ffff"
        "\u2640-\u2642"
        "\u2600-\u2B55"
        "\u23cf"
        "\u23e9"
        "\u231a"
        "\ufe0f"  # variation selector
        "\u3030"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub('', text)

print("✓ Punctuation & quotation normalization functions defined")

✓ Punctuation & quotation normalization functions defined


In [28]:
# ============================================================
#  STEP 2D: Whitespace Normalization & Special Token Injection
# ============================================================

def normalize_whitespace(text):
    """Normalize whitespace: collapse multiple spaces, normalize newlines,
    preserve paragraph structure."""
    # Normalize line endings
    text = re.sub(r'\r\n', '\n', text)
    text = re.sub(r'\r', '\n', text)
    
    # Collapse multiple blank lines into double newline (paragraph break)
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    # Collapse multiple spaces/tabs into single space
    text = re.sub(r'[ \t]+', ' ', text)
    
    # Remove spaces at start/end of lines
    text = re.sub(r' *\n *', '\n', text)
    
    # Strip overall
    text = text.strip()
    
    return text


def inject_special_tokens(text):
    """Replace structural elements with explicit special tokens.
    This is critical for trigram model — explicit tokens help learn transitions."""
    
    # Paragraph breaks → <PARA>
    text = text.replace('\n\n', ' <PARA> ')
    
    # Single newlines → <NL>
    text = text.replace('\n', ' <NL> ')
    
    # Collapse any resulting multiple spaces
    text = re.sub(r' +', ' ', text)
    
    # Wrap the entire story with BOS and EOS
    text = f'<BOS> {text.strip()} <EOS>'
    
    return text

print("✓ Whitespace & special token functions defined")

✓ Whitespace & special token functions defined


In [29]:
# ============================================================
#  STEP 2E: Full Pipeline — Combine All Steps
# ============================================================

def preprocess_story(text, source):
    """Full preprocessing pipeline for a single story."""
    
    # 1. Source-specific artifact removal
    if source == 'adabiduniya.com':
        text = remove_adabi_artifacts(text)
    elif source == 'milkystory.com':
        text = remove_milky_artifacts(text)
    elif source == 'urdupoint.com':
        text = remove_urdupoint_artifacts(text)
    
    # 2. Remove predominantly English lines
    text = remove_english_titles(text)
    
    # 3. Remove emojis
    text = remove_emojis(text)
    
    # 4. Unicode NFC normalization
    text = normalize_unicode(text)
    
    # 5. Urdu character normalization
    text = normalize_urdu_chars(text)
    
    # 6. Clean zero-width characters
    text = clean_zero_width(text)
    
    # 7. Normalize quotation marks
    text = normalize_quotations(text)
    
    # 8. Clean punctuation
    text = clean_punctuation(text)
    
    # 9. Normalize whitespace (before special tokens)
    text = normalize_whitespace(text)
    
    # 10. Inject special tokens
    text = inject_special_tokens(text)
    
    return text


def is_valid_story(processed_text, min_tokens=50):
    """Quality filter: reject stories that are too short or mostly English after cleaning."""
    # Remove special tokens for counting
    clean = re.sub(r'<\w+>', '', processed_text).strip()
    
    # Too short
    tokens = clean.split()
    if len(tokens) < min_tokens:
        return False
    
    # Too much English content remaining
    eng_chars = len(re.findall(r'[a-zA-Z]', clean))
    total = len(clean.replace(' ', ''))
    if total > 0 and eng_chars / total > 0.4:
        return False
    
    return True

print("✓ Full pipeline defined")

✓ Full pipeline defined


## Step 3: Run the Pipeline on All Data

In [30]:
# Combine all sources and run pipeline
all_stories = []
for dataset, source_name in [(adabi, 'adabiduniya.com'), (milky, 'milkystory.com'), (urdupoint, 'urdupoint.com')]:
    for story in dataset:
        all_stories.append({
            'title': story['title'],
            'content': story['content'],
            'source': source_name
        })

print(f"Total raw stories: {len(all_stories)}")

# Process all stories
processed_stories = []
rejected = {'short': 0, 'english_heavy': 0}

for story in all_stories:
    processed = preprocess_story(story['content'], story['source'])
    
    if is_valid_story(processed):
        processed_stories.append({
            'title': story['title'],
            'source': story['source'],
            'text': processed
        })
    else:
        # Track why rejected
        clean = re.sub(r'<\w+>', '', processed).strip()
        tokens = clean.split()
        if len(tokens) < 50:
            rejected['short'] += 1
        else:
            rejected['english_heavy'] += 1

print(f"\nAfter preprocessing & filtering:")
print(f"  Accepted: {len(processed_stories)} stories")
print(f"  Rejected (too short):    {rejected['short']}")
print(f"  Rejected (English-heavy): {rejected['english_heavy']}")
print(f"\nSource distribution:")
source_counts = Counter(s['source'] for s in processed_stories)
for src, count in source_counts.most_common():
    print(f"  {src}: {count}")

Total raw stories: 1113

After preprocessing & filtering:
  Accepted: 796 stories
  Rejected (too short):    317
  Rejected (English-heavy): 0

Source distribution:
  urdupoint.com: 526
  milkystory.com: 192
  adabiduniya.com: 78


In [31]:
# Inspect a few processed examples to verify quality
for i, story in enumerate(processed_stories[:3]):
    print(f"\n{'='*60}")
    print(f"  [{story['source']}] {story['title']}")
    print(f"{'='*60}")
    # Show first 500 chars
    print(story['text'][:500])
    print("...")
    # Show last 200 chars
    print(f"\n...{story['text'][-200:]}")


  [adabiduniya.com] دودھ گر گیا: ناکامی سے اٹھنے کی ایک دل کو چھو لینے والی موٹیویشنل کہانی
<BOS> ایک چھوٹے سے گاؤں میں، سبز پہاڑیوں اور بہتی ندیوں کے درمیان، روہن نامی ایک 10 سال کا لڑکا اپنی بیوہ ماں کاملا کے ساتھ ایک مٹی کے چھوٹے سے گھر میں رہتا تھا۔ کاملا روزانہ امیر گھرانوں کے لیے کپڑے دھوتی، اس کے ہاتھ سخت اور تھکے ہوئے تھے، مگر روہن سے اس کی محبت کبھی کم نہ ہوتی۔ دونوں کے پاس بہت کم تھا، مگر ایک دوسرے کا ساتھ تھا — اور یہی کافی خوشی کے لیے کافی تھا۔ <PARA> ہر صبح روہن سورج نکلنے سے پہلے اٹھتا، اپنی پرانی بھینس گوری کو ندی کے کنارے چراتا، اور شام کو دودھ نکال کر مٹی کے گھڑے میں م
...

... ہمیں یاد دلاتی ہے کہ زندگی میں نقصان ہوتے رہتے ہیں، مگر جو اٹھ کھڑا ہوتا ہے، وہی جیتتا ہے۔ یہ ایک دل کو چھو لینے والی موٹیویشنل کہانی ہے جو بتاتی ہے کہ "ناکامی کا آنسو نہیں، عمل کا قدم" اہم ہے۔ <EOS>

  [adabiduniya.com] The Tortoise and the Hare Story In Urdu A Timeless Tale of Patience and Perseverance
<BOS> یہ ویڈیو ایک کلاسک Aesop کی کہانی کا خوبصورت اینیمیٹڈ ری ٹیلنگ ہے جو صبر، ہمت، تواضع 

## Step 4: Corpus Statistics & Validation

In [32]:
# Corpus-level statistics
full_corpus = '\n'.join(s['text'] for s in processed_stories)

total_chars = len(full_corpus)
total_tokens = len(full_corpus.split())
total_stories = len(processed_stories)

# Count special tokens
bos_count = full_corpus.count('<BOS>')
eos_count = full_corpus.count('<EOS>')
para_count = full_corpus.count('<PARA>')
nl_count = full_corpus.count('<NL>')

# Character distribution (top 30 non-space characters)
char_freq = Counter(c for c in full_corpus if c != ' ' and c != '\n')
top_chars = char_freq.most_common(30)

# Check for remaining problematic Unicode
problematic = Counter()
for c in full_corpus:
    cp = ord(c)
    if cp > 127 and unicodedata.category(c) in ('Cf', 'Cn', 'Co'):
        problematic[f'U+{cp:04X} ({unicodedata.name(c, "UNKNOWN")})'] += 1

print(f"{'='*50}")
print(f"  CORPUS STATISTICS")
print(f"{'='*50}")
print(f"  Stories:          {total_stories}")
print(f"  Total characters: {total_chars:,}")
print(f"  Total tokens:     {total_tokens:,} (whitespace-split)")
print(f"  Avg chars/story:  {total_chars // total_stories:,}")
print(f"  Avg tokens/story: {total_tokens // total_stories:,}")
print(f"\n  Special Tokens:")
print(f"    <BOS>:  {bos_count}")
print(f"    <EOS>:  {eos_count}")
print(f"    <PARA>: {para_count}")
print(f"    <NL>:   {nl_count}")
print(f"\n  Top 30 characters:")
for char, count in top_chars:
    name = unicodedata.name(char, '?')
    print(f"    '{char}' (U+{ord(char):04X} {name}): {count:,}")

if problematic:
    print(f"\n  ⚠ Remaining problematic Unicode chars:")
    for char_info, count in problematic.most_common():
        print(f"    {char_info}: {count}")
else:
    print(f"\n  ✓ No problematic Unicode characters found")

  CORPUS STATISTICS
  Stories:          796
  Total characters: 1,314,221
  Total tokens:     287,519 (whitespace-split)
  Avg chars/story:  1,651
  Avg tokens/story: 361

  Special Tokens:
    <BOS>:  796
    <EOS>:  796
    <PARA>: 4806
    <NL>:   0

  Top 30 characters:
    'ا' (U+0627 ARABIC LETTER ALEF): 118,155
    'ی' (U+06CC ARABIC LETTER FARSI YEH): 98,706
    'ک' (U+06A9 ARABIC LETTER KEHEH): 65,001
    'ر' (U+0631 ARABIC LETTER REH): 60,284
    'و' (U+0648 ARABIC LETTER WAW): 60,105
    'ہ' (U+06C1 ARABIC LETTER HEH GOAL): 52,343
    'ے' (U+06D2 ARABIC LETTER YEH BARREE): 52,289
    'ن' (U+0646 ARABIC LETTER NOON): 48,334
    'م' (U+0645 ARABIC LETTER MEEM): 38,499
    'ت' (U+062A ARABIC LETTER TEH): 38,486
    'س' (U+0633 ARABIC LETTER SEEN): 34,788
    'ل' (U+0644 ARABIC LETTER LAM): 29,778
    'ب' (U+0628 ARABIC LETTER BEH): 29,144
    'ھ' (U+06BE ARABIC LETTER HEH DOACHASHMEE): 28,244
    'د' (U+062F ARABIC LETTER DAL): 25,738
    'ں' (U+06BA ARABIC LETTER NOON GHUNNA):

In [33]:
# Verify no Arabic-form characters remain
test_corpus = full_corpus
remaining_arabic_yeh = test_corpus.count('\u064A')
remaining_arabic_kaf = test_corpus.count('\u0643')

print("Unicode Normalization Verification:")
print(f"  Arabic Yeh (ي) remaining: {remaining_arabic_yeh} {'✓' if remaining_arabic_yeh == 0 else '✗'}")
print(f"  Arabic Kaf (ك) remaining: {remaining_arabic_kaf} {'✓' if remaining_arabic_kaf == 0 else '✗'}")
print(f"  ZWJ remaining:  {test_corpus.count(chr(0x200D))} {'✓' if test_corpus.count(chr(0x200D)) == 0 else '✗'}")
print(f"  ZWNJ remaining: {test_corpus.count(chr(0x200C))} {'✓' if test_corpus.count(chr(0x200C)) == 0 else '✗'}")

# Verify special token structure
print(f"\nStructural Verification:")
print(f"  Every story starts with <BOS>: {all(s['text'].startswith('<BOS>') for s in processed_stories)} ✓")
print(f"  Every story ends with <EOS>:   {all(s['text'].endswith('<EOS>') for s in processed_stories)} ✓")

# Sample token distribution around <PARA>
print(f"\nSample transition around <PARA> (first occurrence):")
idx = full_corpus.find('<PARA>')
if idx > 0:
    print(f"  ...{full_corpus[max(0,idx-60):idx+70]}...")

Unicode Normalization Verification:
  Arabic Yeh (ي) remaining: 0 ✓
  Arabic Kaf (ك) remaining: 0 ✓
  ZWJ remaining:  0 ✓
  ZWNJ remaining: 0 ✓

Structural Verification:
  Every story starts with <BOS>: True ✓
  Every story ends with <EOS>:   True ✓

Sample transition around <PARA> (first occurrence):
  ... ایک دوسرے کا ساتھ تھا — اور یہی کافی خوشی کے لیے کافی تھا۔ <PARA> ہر صبح روہن سورج نکلنے سے پہلے اٹھتا، اپنی پرانی بھینس گوری کو ...


## Step 5: Save the Preprocessed Corpus

Outputs:
- **`corpus.txt`** — single file, one `<BOS>...<EOS>` story per line (for BPE training & trigram model)
- **`corpus.json`** — structured JSON with metadata (source, title) for reference

In [34]:
# # Save corpus.txt — one story per line (BOS...EOS)
# corpus_txt_path = OUTPUT_DIR / "corpus.txt"
# with open(corpus_txt_path, "w", encoding="utf-8") as f:
#     for story in processed_stories:
#         f.write(story['text'] + '\n')

# # Save corpus.json — structured with metadata
# corpus_json_path = OUTPUT_DIR / "corpus.json"
# with open(corpus_json_path, "w", encoding="utf-8") as f:
#     json.dump(processed_stories, f, ensure_ascii=False, indent=2)

# # File sizes
# txt_size = os.path.getsize(corpus_txt_path)
# json_size = os.path.getsize(corpus_json_path)

print(f"Saved preprocessed corpus:")
# print(f"  {corpus_txt_path}  ({txt_size / 1024:.1f} KB)")
# print(f"  {corpus_json_path} ({json_size / 1024:.1f} KB)")
print(f"\n  Stories: {len(processed_stories)}")
print(f"  Format:  One <BOS>...<EOS> story per line in corpus.txt")
print(f"\nSpecial tokens available for BPE vocabulary:")
print(f"  <BOS>  — Beginning of story")
print(f"  <EOS>  — End of story")
print(f"  <PAD>  — Padding (add during batching)")
print(f"  <UNK>  — Unknown token (add during BPE)")
print(f"  <PARA> — Paragraph break")
print(f"  <NL>   — Single newline")

Saved preprocessed corpus:

  Stories: 796
  Format:  One <BOS>...<EOS> story per line in corpus.txt

Special tokens available for BPE vocabulary:
  <BOS>  — Beginning of story
  <EOS>  — End of story
  <PAD>  — Padding (add during batching)
  <UNK>  — Unknown token (add during BPE)
  <PARA> — Paragraph break
  <NL>   — Single newline


## Step 6: Identify & Separate Islamic Stories

In [35]:
# Identify Islamic/religious stories using keyword matching on titles AND content
# We check both raw titles and raw content from all_stories (pre-processed originals)

islamic_keywords_title = [
    # Urdu keywords for titles
    'رمضان', 'روزہ', 'روزے', 'نماز', 'حج', 'زکوٰۃ', 'زکات',
    'قرآن', 'حدیث', 'سورۃ', 'سورہ', 'آیت', 'آیات',
    'اسلام', 'اسلامی', 'مسلم', 'مسلمان', 
    'اللہ', 'محمد ﷺ', 'نبی', 'رسول', 'پیغمبر',
    'صحابہ', 'صحابی', 'خلیفہ', 'خلفاء',
    'مسجد', 'کعبہ', 'مدینہ', 'مکہ',
    'جنت', 'جہنم', 'قیامت', 'آخرت',
    'تقویٰ', 'توبہ', 'شہادت', 'جہاد', 'غزوہ', 'بدر',
    'عید', 'شعبان', 'محرم', 'عاشورہ',
    'فرض', 'سنت', 'مستحب', 'حرام', 'حلال',
    'شریعت', 'فقہ', 'تفسیر',
    'عمرہ', 'اعتکاف', 'افطار', 'سحری',
]

islamic_keywords_content = [
    # Strong indicators in content (need multiple hits to classify)
    'صلی اللہ علیہ وسلم', 'ﷺ', 'علیہ السلام', 'رضی اللہ عنہ',
    'سبحان اللہ', 'الحمدللہ', 'ماشاءاللہ', 'انشاءاللہ',
    'قرآن مجید', 'قرآن کریم', 'حدیث شریف', 'صحیح بخاری', 'صحیح مسلم',
    'سورۃ البقرہ', 'سورۃ الفاتحہ', 'سورۃ',
    'رمضان المبارک', 'ماہ رمضان',
    'غزوہ بدر', 'غزوہ احد', 'غزوہ خندق',
    'فرضیت', 'نماز پنجگانہ',
]

def is_islamic_story(title, content):
    """Classify a story as Islamic based on title and content keywords."""
    title_lower = title.strip()
    
    # Title match — any keyword in title is strong signal
    for kw in islamic_keywords_title:
        if kw in title_lower:
            return True, f'title: "{kw}"'
    
    # Content match — need multiple strong indicators
    content_hits = []
    for kw in islamic_keywords_content:
        if kw in content:
            content_hits.append(kw)
    
    if len(content_hits) >= 3:
        return True, f'content: {len(content_hits)} hits ({", ".join(content_hits[:3])}...)'
    
    # Check density of broader Islamic terms in content
    broad_terms = ['اللہ', 'نبی', 'رسول', 'قرآن', 'حدیث', 'روزہ', 'نماز', 
                   'مسجد', 'صحابہ', 'جنت', 'جہنم', 'فرض', 'سنت', 'تقویٰ',
                   'رمضان', 'شریعت', 'توحید', 'رسالت', 'آخرت']
    broad_hits = sum(1 for kw in broad_terms if kw in content)
    # If many broad terms + reasonable density, it's Islamic
    if broad_hits >= 6:
        return True, f'content: {broad_hits} broad Islamic terms'
    
    return False, ''

# Scan all raw stories (before preprocessing)
islamic_indices = []
non_islamic_indices = []

print(f"Scanning {len(all_stories)} stories for Islamic content...\n")
print(f"{'#':>3}  {'Source':<20} {'Title':<50} {'Reason'}")
print(f"{'-'*3}  {'-'*20} {'-'*50} {'-'*30}")

for i, story in enumerate(all_stories):
    is_islamic, reason = is_islamic_story(story['title'], story['content'])
    if is_islamic:
        islamic_indices.append(i)
        print(f"{i:>3}  {story['source']:<20} {story['title'][:48]:<50} {reason}")

print(f"\n{'='*60}")
print(f"  Islamic stories found: {len(islamic_indices)} / {len(all_stories)}")
print(f"  Non-Islamic stories:   {len(all_stories) - len(islamic_indices)}")
print(f"{'='*60}")

Scanning 1113 stories for Islamic content...

  #  Source               Title                                              Reason
---  -------------------- -------------------------------------------------- ------------------------------
  4  adabiduniya.com      روزہ اسلام سے پہلے بھی تھا                         title: "روزہ"
  9  adabiduniya.com      غیبت کا گھونٹ: غیبت پر پیاری اسلامی کہانی بچوں ک   title: "اسلام"
 10  adabiduniya.com      تین چور اور تین قبروں کا قصہ: توبہ اور صدقہ کی ب   title: "توبہ"
 12  adabiduniya.com      اللہ سے دوستی: رمضان کی تیاری – ایک دل کو چھو لی   title: "رمضان"
 13  adabiduniya.com      عید کا امتحان: اللہ سب ٹھیک کر دے گا – ایک دل کو   title: "اسلام"
 17  adabiduniya.com      بسم اللہ اور شیطان: اللہ کی حفاظت – ایک پیاری اس   title: "اسلام"
 18  adabiduniya.com      شیطان کی موت کیسے ہوگی اور اسے کون دفن کرے گا؟ –   title: "قرآن"
 19  adabiduniya.com      شیطان کی چال: اعوذ باللہ کا کمال – ایک دل کو چھو   title: "اسلام"
 20  adabiduniya.com      بسم

In [36]:


# False positive exclusion patterns (names, places containing Islamic keywords)
false_positive_patterns = [
    'احسان اللہ', 'شجاعت اللہ', 'نصر اللہ', 'عنایت اللہ',
    'حبیب اللہ', 'عبداللہ', 'اسلام آباد', 'فیصل آباد',
    'انضمام الحق', 'مصباح الحق',
]

def is_islamic_story_refined(title, content):
    """Refined classification — avoids false positives from names/places."""
    
    # Check if title contains a false positive pattern
    for fp in false_positive_patterns:
        if fp in title:
            # The Islamic keyword is part of a name/place — don't match on title alone
            # Still check content though
            break
    else:
        # No false positive in title — check title keywords with word boundaries
        # Using space/start/end as word boundaries for Urdu
        title_words = set(title.split())
        
        # Strong title indicators (exact word match)
        strong_title_kw = [
            'رمضان', 'روزہ', 'روزے', 'نماز', 'حج', 'زکوٰۃ',
            'قرآن', 'حدیث', 'سورۃ', 'سورہ',
            'اسلامی', 'صحابہ', 'صحابی', 'خلیفہ',
            'مسجد', 'کعبہ', 'تقویٰ', 'شہادت', 'غزوہ',
            'شعبان', 'محرم', 'عاشورہ', 'افطار', 'سحری',
            'اعتکاف', 'شریعت', 'فقہ', 'تفسیر', 'عمرہ',
        ]
        for kw in strong_title_kw:
            if kw in title:
                return True, f'title: "{kw}"'
        
        # Phrase-level title checks (more than just a keyword)
        title_phrases = [
            'اللہ کی', 'اللہ سے', 'اللہ پر', 'اللہ کا', 'اللہ نے', 'اللہ کے',
            'بسم اللہ', 'نبی ﷺ', 'نبی کریم', 'رسول اللہ',
            'پیاری اسلامی', 'اسلامی کہانی', 'اسلام میں', 'اسلام سے',
            'نماز کی', 'نماز یا', 'توبہ اور', 'توبہ کی',
            'ایمان کی', 'ایمان اور',
            'حضرت', 'علیہ السلام',
        ]
        for phrase in title_phrases:
            if phrase in title:
                return True, f'title: "{phrase}"'
    
    # Content-level check — need strong density of Islamic terms
    strong_content_markers = [
        'صلی اللہ علیہ وسلم', 'ﷺ', 'علیہ السلام', 'رضی اللہ عنہ',
        'قرآن مجید', 'قرآن کریم', 'حدیث شریف', 'صحیح بخاری', 'صحیح مسلم',
        'رمضان المبارک', 'ماہ رمضان', 'نماز پنجگانہ',
        'غزوہ بدر', 'غزوہ احد', 'غزوہ خندق',
        'سبحان اللہ', 'الحمدللہ', 'سورۃ البقرہ', 'سورۃ الفاتحہ',
        'فرضیت', 'روزے فرض', 'نماز فرض',
    ]
    strong_hits = sum(1 for kw in strong_content_markers if kw in content)
    if strong_hits >= 3:
        return True, f'content: {strong_hits} strong Islamic markers'
    
    # Broad content terms density check
    broad = ['اللہ', 'نبی', 'رسول', 'قرآن', 'حدیث', 'روزہ', 'نماز', 
             'صحابہ', 'جنت', 'جہنم', 'فرض', 'سنت', 'تقویٰ',
             'رمضان', 'شریعت', 'توحید', 'آخرت', 'ثواب', 'گناہ']
    broad_hits = sum(1 for kw in broad if kw in content)
    if broad_hits >= 8:
        return True, f'content: {broad_hits} broad Islamic terms'
    
    return False, ''

# Re-scan
islamic_stories_refined = []
non_islamic_stories_refined = []

print(f"Refined scan of {len(all_stories)} stories...\n")
print(f"{'#':>3}  {'Source':<20} {'Title':<55} {'Reason'}")
print(f"{'-'*3}  {'-'*20} {'-'*55} {'-'*30}")

for i, story in enumerate(all_stories):
    is_islamic, reason = is_islamic_story_refined(story['title'], story['content'])
    if is_islamic:
        islamic_stories_refined.append(i)
        print(f"{i:>3}  {story['source']:<20} {story['title'][:53]:<55} {reason}")
    else:
        non_islamic_stories_refined.append(i)

print(f"\n{'='*60}")
print(f"  Islamic stories:     {len(islamic_stories_refined)} / {len(all_stories)}")
print(f"  Non-Islamic stories: {len(non_islamic_stories_refined)} / {len(all_stories)}")
print(f"{'='*60}")

Refined scan of 1113 stories...

  #  Source               Title                                                   Reason
---  -------------------- ------------------------------------------------------- ------------------------------
  4  adabiduniya.com      روزہ اسلام سے پہلے بھی تھا                              title: "روزہ"
  9  adabiduniya.com      غیبت کا گھونٹ: غیبت پر پیاری اسلامی کہانی بچوں کے لیے   title: "اسلامی"
 10  adabiduniya.com      تین چور اور تین قبروں کا قصہ: توبہ اور صدقہ کی برکت –   title: "توبہ اور"
 12  adabiduniya.com      اللہ سے دوستی: رمضان کی تیاری – ایک دل کو چھو لینے وا   title: "رمضان"
 13  adabiduniya.com      عید کا امتحان: اللہ سب ٹھیک کر دے گا – ایک دل کو چھو    title: "اسلامی"
 17  adabiduniya.com      بسم اللہ اور شیطان: اللہ کی حفاظت – ایک پیاری اسلامی    title: "اسلامی"
 18  adabiduniya.com      شیطان کی موت کیسے ہوگی اور اسے کون دفن کرے گا؟ – قرآن   title: "قرآن"
 19  adabiduniya.com      شیطان کی چال: اعوذ باللہ کا کمال – ایک دل کو چھو لینے   

In [37]:
# Separate Islamic stories from processed_stories and rebuild corpus
islamic_titles = set()
for i in islamic_stories_refined:
    islamic_titles.add(all_stories[i]['title'])

# Split processed_stories into Islamic and non-Islamic
islamic_processed = [s for s in processed_stories if s['title'] in islamic_titles]
non_islamic_processed = [s for s in processed_stories if s['title'] not in islamic_titles]

print(f"Processed stories breakdown:")
print(f"  Total processed:  {len(processed_stories)}")
print(f"  Islamic:          {len(islamic_processed)}")
print(f"  Non-Islamic:      {len(non_islamic_processed)}")
print(f"  Sum check:        {len(islamic_processed) + len(non_islamic_processed)}")

# Show source distribution for each
print(f"\nIslamic by source:")
for src, cnt in Counter(s['source'] for s in islamic_processed).most_common():
    print(f"  {src}: {cnt}")

print(f"\nNon-Islamic by source:")
for src, cnt in Counter(s['source'] for s in non_islamic_processed).most_common():
    print(f"  {src}: {cnt}")

Processed stories breakdown:
  Total processed:  796
  Islamic:          39
  Non-Islamic:      757
  Sum check:        796

Islamic by source:
  adabiduniya.com: 21
  urdupoint.com: 16
  milkystory.com: 2

Non-Islamic by source:
  urdupoint.com: 510
  milkystory.com: 190
  adabiduniya.com: 57


In [38]:
print(f"    Stories: {len(non_islamic_processed)}")
print(f"    Stories: {len(islamic_processed)}")
print(f"\n  Islamic stories removed from main corpus ✓")

    Stories: 757
    Stories: 39

  Islamic stories removed from main corpus ✓


## Step 7: Remove News & Promotional Articles — Keep Only Stories

In [39]:
# ============================================================
# NEWS & PROMOTIONAL ARTICLE DETECTION
# ============================================================
# Strategy:
# - milkystory.com: WHITELIST approach (most are news; identify actual stories by URL/title keywords)
# - adabiduniya.com: BLACKLIST approach (most are stories; remove historical essays/articles)
# - urdupoint.com: KEEP ALL (all are kids stories)

def get_raw_url(story, raw_data):
    """Get original URL from raw data by matching title."""
    for raw in raw_data:
        if raw['title'] == story['title']:
            return raw.get('url', '')
    return ''

def get_url_slug(url):
    """Extract the URL path/slug after the domain, lowercase."""
    from urllib.parse import urlparse
    parsed = urlparse(url)
    return parsed.path.lower()

def is_actual_story_milky(story, raw_data):
    """Identify actual stories from milkystory.com (vs news/promo articles).
    Uses URL slug keywords and title patterns to whitelist stories."""
    url = get_raw_url(story, raw_data)
    title = story['title']
    slug = get_url_slug(url)  # Path only, no domain
    
    # URL slug-based story indicators (checks path, not domain)
    story_slug_keywords = [
        'kahani', '-story', 'afsana', 'novel', 'manto',
        'qist',  # serial episodes
    ]
    for kw in story_slug_keywords:
        if kw in slug:
            return True
    
    # Title-based story indicators — be STRICT, avoid words used in news loosely
    story_title_keywords = [
        'افسانہ', 'ناول', 'قسط',
        'داستان', 'حکایت',
    ]
    for kw in story_title_keywords:
        if kw in title:
            return True
    
    # Known literary works: use STARTS-WITH matching to avoid false positives
    # (e.g. 'قرض' matching "قرض کی فراہمی" in a news title)
    known_stories_startswith = [
        'بُو', 'ٹھنڈا گوشت', 'کھول دو', 'کالی شلوار',
        'گڑیا کی شادی', 'انوکھی محبت', 'کام چور',
        'جن کا بچہ', 'میٹھے جوتے', 'بکھرے خوابوں کی تعبیر',
        'پورے چاند کی رات', 'قرض(',  # "قرض(Qarz)" not "قرض کی فراہمی"
    ]
    for known in known_stories_startswith:
        if title.startswith(known):
            return True
    
    return False

def is_non_story_adabi(story, raw_data):
    """Identify non-story content from adabiduniya.com (historical essays, articles, informational)."""
    url = get_raw_url(story, raw_data)
    title = story['title']
    slug = get_url_slug(url)
    
    # URL-based non-story indicators
    non_story_url_keywords = [
        'history', 'essay', 'hub-ul-watni',
        'wolf-mentality',  # self-help article, not a story
    ]
    for kw in non_story_url_keywords:
        if kw in slug:
            return True
    
    # Historical/biographical articles by title keywords
    historical_keywords = [
        'تاریخی پس منظر', 'تاریخ کا', 'یورپ کا فاتح',
        'عظیم فلسفی', 'عظیم فاتح', 'ہیرو یا ولن',
        'اہرامات کے راز', 'سفاک حکمران',
        'ایشیا کی جنگ',  # Alexander the Great article
    ]
    for kw in historical_keywords:
        if kw in title:
            return True
    
    # Informational / essay articles
    info_keywords = [
        'کیا ہے؟',  # "What is...?"
        'حب الوطنی',  # patriotism essay
    ]
    for kw in info_keywords:
        if kw in title:
            return True
    
    # English-only titles (failed preprocessing artifacts)
    if re.match(r'^[A-Za-z\s]+$', title.strip()):
        return True
    
    return False

def is_news_or_promotional(story, raw_data_map):
    """Master classifier: returns (is_news, reason) tuple."""
    source = story['source']
    
    if source == 'urdupoint.com':
        return False, 'kids story (urdupoint)'
    
    if source == 'milkystory.com':
        if is_actual_story_milky(story, raw_data_map['milky']):
            return False, 'actual story (milky whitelist)'
        return True, 'news/promo article (milky)'
    
    if source == 'adabiduniya.com':
        if is_non_story_adabi(story, raw_data_map['adabi']):
            return True, 'non-story article (adabi)'
        return False, 'actual story (adabi)'
    
    return False, 'unknown source (kept)'

# Apply classification
raw_data_map = {'milky': milky, 'adabi': adabi, 'urdupoint': urdupoint}

stories_keep = []
stories_remove = []

for story in processed_stories:
    is_news, reason = is_news_or_promotional(story, raw_data_map)
    if is_news:
        stories_remove.append({**story, '_reason': reason})
    else:
        stories_keep.append({**story, '_reason': reason})

# Summary
print(f"Total processed stories: {len(processed_stories)}")
print(f"Stories to KEEP: {len(stories_keep)}")
print(f"Stories to REMOVE: {len(stories_remove)}")
print()

# Breakdown by source
for src in ['milkystory.com', 'adabiduniya.com', 'urdupoint.com']:
    kept = [s for s in stories_keep if s['source'] == src]
    removed = [s for s in stories_remove if s['source'] == src]
    print(f"{src}:")
    print(f"  Kept: {len(kept)}, Removed: {len(removed)}")

print()
print("KEPT milkystory titles:")
for s in stories_keep:
    if s['source'] == 'milkystory.com':
        print(f"  + {s['title'][:80]}")

print()
print("REMOVED adabiduniya titles:")
for s in stories_remove:
    if s['source'] == 'adabiduniya.com':
        print(f"  - {s['title'][:80]}")

Total processed stories: 796
Stories to KEEP: 606
Stories to REMOVE: 190

milkystory.com:
  Kept: 16, Removed: 176
adabiduniya.com:
  Kept: 64, Removed: 14
urdupoint.com:
  Kept: 526, Removed: 0

KEPT milkystory titles:
  + انوکھی محبت
  + سانتا کلاز کی کہانی: یہ کردار کس نے تخلیق کیا؟
  + بُو
  + کام چور
  + قرض(Qarz)
  + بکھرے خوابوں کی تعبیر
  + ٹھنڈا گوشت
  + گڑیا کی شادی
  + بکھرے خوابوں کی تعبیر
  + جن کا بچہ
  + پورے چاند کی رات
  + کھول دو
  + بکھرے خوابوں کی تعبیر
  + بکھرے خوابوں کی تعبیر
  + کالی شلوار
  + میٹھے جوتے

REMOVED adabiduniya titles:
  - The Tortoise and the Hare Story In Urdu A Timeless Tale of Patience and Persever
  - عظیم دیوار کیسے بنی؟ – تاریخی پس منظر
  - ویلنٹائن ڈے کیا ہے؟
  - وولف مینٹیلٹی: بھیڑیے کی 8 طاقتور اصول جو آپ کو زندگی کا بادشاہ بنا دیں گے
  - Easy Short Story In Urdu
  - Short Story In Urdu
  - عنوان: حب الوطنی – ایمان کا حصہ اور قوم کی طاقت
  - Seven Princess Short Urdu Story
  - نپولین بوناپارٹ: یورپ کا فاتح یا شرپسند؟
  - سقراط ایتھنز کا ع

In [40]:
# ============================================================
# Save final corpus — stories only (no news, no promotional, no Islamic)
# ============================================================

# Clean up: remove internal _reason key
final_stories = []
for s in stories_keep:
    clean = {k: v for k, v in s.items() if not k.startswith('_')}
    final_stories.append(clean)

print(f"Final corpus: {len(final_stories)} stories")
print(f"  milkystory.com: {sum(1 for s in final_stories if s['source'] == 'milkystory.com')}")
print(f"  adabiduniya.com: {sum(1 for s in final_stories if s['source'] == 'adabiduniya.com')}")
print(f"  urdupoint.com: {sum(1 for s in final_stories if s['source'] == 'urdupoint.com')}")

# Save as corpus.txt (one story per line — tokenized text with special tokens)
corpus_txt_path = OUTPUT_DIR / 'corpus.txt'
with open(corpus_txt_path, 'w', encoding='utf-8') as f:
    for story in final_stories:
        f.write(story['text'] + '\n')

# Save as corpus.json (with metadata)
corpus_json_path = OUTPUT_DIR / 'corpus.json'
dataset = []
for story in final_stories:
    dataset.append({
        'title': story['title'],
        'source': story['source'],
        'text': story['text'],
    })

with open(corpus_json_path, 'w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

# File sizes
txt_size = os.path.getsize(corpus_txt_path)
json_size = os.path.getsize(corpus_json_path)
print(f"\nSaved:")
print(f"  {corpus_txt_path} ({txt_size/1024:.1f} KB)")
print(f"  {corpus_json_path} ({json_size/1024:.1f} KB)")

# Quick stats
total_tokens = sum(len(s['text'].split()) for s in final_stories)
total_chars = sum(len(s['text']) for s in final_stories)
print(f"\nCorpus stats:")
print(f"  Total stories: {len(final_stories)}")
print(f"  Total tokens: {total_tokens:,}")
print(f"  Total characters: {total_chars:,}")
print(f"  Avg tokens/story: {total_tokens/len(final_stories):.0f}")
print(f"  Avg chars/story: {total_chars/len(final_stories):.0f}")

# Clean up temp analysis files
for tmp in ['_analysis.txt', '_classification.txt']:
    p = OUTPUT_DIR / tmp
    if p.exists():
        p.unlink()
        print(f"  Cleaned up {tmp}")

Final corpus: 606 stories
  milkystory.com: 16
  adabiduniya.com: 64
  urdupoint.com: 526

Saved:
  ../data/corpus.txt (1727.4 KB)
  ../data/corpus.json (1793.5 KB)

Corpus stats:
  Total stories: 606
  Total tokens: 225,251
  Total characters: 1,016,845
  Avg tokens/story: 372
  Avg chars/story: 1678
